In [ ]:
# ============================================================
# BCI Competition IV Dataset 2b — STSPec Model
# Adapted from BCI IV 2a version
# Key differences from 2a:
#   - 3 EEG channels  (vs 22)
#   - 2 classes: Left hand / Right hand  (vs 4)
#   - File prefix B0{s}T / B0{s}E  (vs A0{s}T / A0{s}E)
# ============================================================

# Cell - Imports
import os
import time
import random

import numpy as np
import pandas as pd
from numpy.linalg import eig
import scipy.io as sio

import tensorflow as tf
from tensorflow import Variable
from tensorflow import keras
from tensorflow.keras.models import Model
from keras.models import load_model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, Callback
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras import layers
from tensorflow.keras.layers import (
    Input, Dense, Activation, Dropout, Conv1D, Conv2D, SeparableConv1D, SeparableConv2D,
    DepthwiseConv2D, AveragePooling2D, MaxPooling1D,
    BatchNormalization, Dropout, Add, Lambda, Permute,
    Reshape, Concatenate, GlobalAveragePooling1D, GlobalMaxPooling1D,
    Multiply, Bidirectional, LSTM, Flatten
)
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.constraints import max_norm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, cohen_kappa_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler

from scipy.signal import butter, filtfilt
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# --- Reproducibility ---
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("All imports successful.")
print(f"TensorFlow version: {tf.__version__}")


# =============================================================
# Cell - Pre-Processing & Data Loading  [CHANGED for 2b]
# =============================================================
def load_BCI2b_data(data_path, subject, training):
    """
    Load one subject's data from BCI Competition IV Dataset 2b.
    Only artifact-free trials are kept (a_artifacts == 0).

    FIX: Uses dynamic list accumulation instead of a pre-allocated array.
    This prevents IndexError for subjects (e.g. 4, 8) whose .mat files
    contain more sessions/runs than a fixed-size array can accommodate.

    t1:t2  →  2 s – 6 s  =  1000 samples @ 250 Hz
    """
    n_channels    = 3
    window_length = 8 * 250   # full window size in the .mat file
    t1 = int(3 * 250)         # 500  — crop start
    t2 = int(7 * 250)         # 1500 — crop end  →  1000-sample segment

    suffix    = "T" if training else "E"
    file_path = os.path.join(data_path, f"B0{subject}{suffix}.mat")

    a      = sio.loadmat(file_path)
    a_data = a["data"]

    # -----------------------------------------------------------------
    # FIX: accumulate into lists — no pre-allocation, no fixed upper bound
    # -----------------------------------------------------------------
    data_list  = []
    class_list = []

    for ii in range(a_data.size):
        a_data1     = a_data[0, ii]
        a_data3     = a_data1[0, 0]
        a_X         = a_data3[0]
        a_trial     = a_data3[1]
        a_y         = a_data3[2]
        a_artifacts = a_data3[5]

        for trial in range(a_trial.size):
            artifact = int(np.array(a_artifacts[trial]).item())
            if artifact == 0:                                     # only valid trials
                start   = int(np.array(a_trial[trial]).item())
                segment = np.transpose(
                    a_X[start:(start + window_length), :n_channels]
                )                                                 # (n_channels, window_length)
                data_list.append(segment[:, t1:t2])              # crop to 1000 samples
                class_list.append(int(np.array(a_y[trial]).item()))

    data_return  = np.array(data_list)                           # (n_valid, 3, 1000)
    class_return = (np.array(class_list) - 1).astype(int)       # 1-based → 0-based

    print(f"  Subject {subject} {'Train' if training else 'Test'}: "
          f"{len(data_list)} artifact-free trials loaded")
    return data_return, class_return


def get_data(path, subject):
    X_train, y_train = load_BCI2b_data(path, subject, training=True)
    X_test,  y_test  = load_BCI2b_data(path, subject, training=False)

    N_tr, N_ch, T = X_train.shape
    X_train = X_train.reshape(N_tr, N_ch, T)

    N_te, N_ch, T = X_test.shape
    X_test  = X_test.reshape(N_te, N_ch, T)

    return X_train, y_train, X_test, y_test


# =============================================================
# Cell - Loading Subjects
# =============================================================
data_path = r"/kaggle/input/datasets/uddeshayanand/bci-iv-2b-dataset"

subjects = [1]   # extend to [1..9] for cross-subject experiments

all_X_train, all_y_train = [], []
all_X_test,  all_y_test  = [], []

for subject in subjects:
    print(f"Loading subject {subject} …")
    X_train, y_train, X_test, y_test = get_data(data_path, subject)
    all_X_train.append(X_train);  all_y_train.append(y_train)
    all_X_test.append(X_test);    all_y_test.append(y_test)

X_train = np.concatenate(all_X_train, axis=0)
y_train = np.concatenate(all_y_train, axis=0)
X_test  = np.concatenate(all_X_test,  axis=0)
y_test  = np.concatenate(all_y_test,  axis=0)

print("X_train shape:", X_train.shape)   # (n, 3, 1000)
print("y_train shape:", y_train.shape)
print("X_test  shape:", X_test.shape)
print("y_test  shape:", y_test.shape)


# =============================================================
# Cell - Data Augmentation (Sliding Window)
# =============================================================

# ------------------------------------------------------------------
# FIX: define sliding-window hyper-params once as module-level
# constants so that n_windows can be computed cleanly from first
# principles instead of back-calculated via integer division (which
# silently drops segments when trial counts differ across subjects).
# ------------------------------------------------------------------
SFREQ       = 250
SW_OVERLAP  = 0.95
SW_SHIFT_SEC = 0.1

SW_SHIFT    = int(round(SW_SHIFT_SEC * SFREQ))                        # 25 samples
SW_WIN_LEN  = int(round(SW_SHIFT / (1.0 - SW_OVERLAP)))               # 500 samples
# n_windows per trial — computed from the raw trial length (1000 samples)
_N_TIMES_RAW = X_train.shape[2]                                        # 1000
N_WINDOWS    = (_N_TIMES_RAW - SW_WIN_LEN) // SW_SHIFT + 1            # 21


def sliding_window(data, sfreq=SFREQ, overlap=SW_OVERLAP, shift_sec=SW_SHIFT_SEC):
    """
    Slide a fixed-length window over each trial.
    Input : (n_trials, n_channels, n_times)
    Output: (n_trials * n_windows, n_channels, window_length)
    """
    n_trials, n_chans, n_times = data.shape
    shift         = int(round(shift_sec * sfreq))
    window_length = int(round(shift / (1.0 - overlap)))
    n_windows     = (n_times - window_length) // shift + 1

    out = np.zeros((n_trials * n_windows, n_chans, window_length), dtype=data.dtype)
    idx = 0
    for tr in range(n_trials):
        for w in range(n_windows):
            start      = w * shift
            out[idx]   = data[tr, :, start:start + window_length]
            idx       += 1
    return out


X_train_seg = sliding_window(X_train)
X_test_seg  = sliding_window(X_test)

# ------------------------------------------------------------------
# FIX: repeat labels using N_WINDOWS computed directly from params,
# not from integer division of segment count — robust for all subjects
# ------------------------------------------------------------------
y_train_seg = np.repeat(y_train, N_WINDOWS)
y_test_seg  = np.repeat(y_test,  N_WINDOWS)

# Sanity checks — these will raise immediately if something is wrong
assert len(y_train_seg) == X_train_seg.shape[0], (
    f"Train label/data mismatch: {len(y_train_seg)} labels vs "
    f"{X_train_seg.shape[0]} segments"
)
assert len(y_test_seg) == X_test_seg.shape[0], (
    f"Test label/data mismatch: {len(y_test_seg)} labels vs "
    f"{X_test_seg.shape[0]} segments"
)

print(f"Segmented X_train_seg : {X_train_seg.shape}, y_train_seg: {y_train_seg.shape}")
print(f"Segmented X_test_seg  : {X_test_seg.shape},  y_test_seg : {y_test_seg.shape}")


# =============================================================
# Cell - Splitting the data
# =============================================================
# Transpose to (samples, time, channels)  →  (N, 500, 3)
X_trainS = X_train_seg.transpose(0, 2, 1)
X_testS  = X_test_seg.transpose(0, 2, 1)

print("X_trainS shape:", X_trainS.shape)
print("X_testS  shape:", X_testS.shape)

X_combined = np.concatenate((X_trainS, X_testS), axis=0)
y_combined = np.concatenate((y_train_seg, y_test_seg), axis=0)
print("Combined:", X_combined.shape, y_combined.shape)

# --- Params ---
batch_size       = 64
epochs           = 1200
validation_split = 0.1
test_split       = 0.2
num_classes      = 2          # Left hand / Right hand

X_trainval, X_test_new, y_trainval, y_test_new = train_test_split(
    X_combined, y_combined,
    test_size=test_split,
    stratify=y_combined,
    random_state=42
)
X_train_new, X_val_new, y_train_new, y_val_new = train_test_split(
    X_trainval, y_trainval,
    test_size=validation_split / (1 - test_split),
    stratify=y_trainval,
    random_state=42
)

# --- One-hot encode labels ---
y_train_new_oh = to_categorical(y_train_new, num_classes)
y_val_new_oh   = to_categorical(y_val_new,   num_classes)
y_test_new_oh  = to_categorical(y_test_new,  num_classes)


# =============================================================
# Cell - Band-pass Filtering
# =============================================================
def apply_band_filters(X_data, fs=250, order=4):
    """
    Input : (samples, timepoints, channels)
    Output: delta, theta, alpha, beta, gamma  —  each (samples, timepoints, channels)
    """
    X_data = X_data - np.mean(X_data, axis=1, keepdims=True)   # remove DC offset

    bands = {
        'delta': (0.5,  4),
        'theta': (4,    8),
        'alpha': (8,   13),
        'beta':  (13,  30),
        'gamma': (30, 100),
    }

    n_samples, n_time, n_channels = X_data.shape
    X_reshaped = X_data.transpose(0, 2, 1).reshape(-1, n_time)  # (N*C, T)

    band_data = {}
    for band_name, (low, high) in bands.items():
        nyq  = 0.5 * fs
        Wn   = [low / nyq, high / nyq]
        if Wn[1] >= 1.0:
            raise ValueError(f"Invalid high cutoff {high} Hz for {fs} Hz sampling rate")
        b, a     = butter(order, Wn, btype='bandpass')
        padlen   = min(150, X_reshaped.shape[1] // 2)
        filtered = filtfilt(b, a, X_reshaped, axis=1, padlen=padlen)
        band_data[band_name] = (
            filtered.reshape(n_samples, n_channels, n_time)
                    .transpose(0, 2, 1)         # → (samples, time, channels)
        )

    return (
        band_data['delta'],
        band_data['theta'],
        band_data['alpha'],
        band_data['beta'],
        band_data['gamma'],
    )


print("Processing frequency bands …")

# apply_band_filters expects (samples, time, channels) — X_*_new is already (N, 500, 3) ✓
_, X_train_theta, X_train_alpha, X_train_beta, _ = apply_band_filters(X_train_new)
_, X_test_theta,  X_test_alpha,  X_test_beta,  _ = apply_band_filters(X_test_new)
_, X_val_theta,   X_val_alpha,   X_val_beta,   _ = apply_band_filters(X_val_new)

print("Band filtering done. X_train_new shape:", X_train_new.shape)   # (N, 500, 3)


# =============================================================
# Cell - Model Helper Classes & Functions
# =============================================================
class Channel_attention(Model):
    def __init__(self, num_ch):
        super(Channel_attention, self).__init__(name='')
        self.num_ch  = num_ch
        self.conv1   = Conv2D(filters=8, kernel_size=(1, 1), kernel_constraint=max_norm(2., axis=(0, 1, 2)))
        self.conv2   = Conv2D(filters=8, kernel_size=(1, 1), kernel_constraint=max_norm(2., axis=(0, 1, 2)))
        self.reshape = Reshape((num_ch, -1))
        self.gamma   = Variable(initial_value=0, dtype='float32', trainable=True)

    def get_config(self):
        config = super().get_config()
        config.update({"num_ch": self.num_ch})
        return config

    def call(self, x):
        proj_query = self.reshape(self.conv1(x))
        proj_key   = self.reshape(self.conv2(x))
        energy     = tf.matmul(proj_query, proj_key, transpose_b=True)
        max_H      = tf.math.reduce_max(energy, axis=2, keepdims=True)
        min_H      = tf.math.reduce_min(energy, axis=2, keepdims=True)
        energy     = (energy - min_H) / ((max_H - min_H) + 1e-8)
        attention  = tf.nn.softmax(energy)
        out = (self.gamma
               * tf.expand_dims(
                   tf.matmul(attention, tf.squeeze(x, axis=-1), transpose_a=True),
                   axis=3)
               + x)
        return out


class Time_attention(Model):
    def __init__(self, num_t):
        super(Time_attention, self).__init__(name='')
        self.num_t   = num_t
        self.conv1   = Conv2D(filters=8, kernel_size=(1, 1), kernel_constraint=max_norm(2., axis=(0, 1, 2)))
        self.conv2   = Conv2D(filters=8, kernel_size=(1, 1), kernel_constraint=max_norm(2., axis=(0, 1, 2)))
        self.reshape = Reshape((num_t, -1))
        self.permute = Permute((2, 1, 3))
        self.gamma   = Variable(initial_value=0, dtype='float32', trainable=True)

    def get_config(self):
        config = super().get_config()
        config.update({"num_t": self.num_t})
        return config

    def call(self, x):
        proj_query = self.reshape(self.permute(self.conv1(x)))
        proj_key   = self.reshape(self.permute(self.conv2(x)))
        energy     = tf.matmul(proj_query, proj_key, transpose_b=True)
        max_H      = tf.math.reduce_max(energy, axis=2, keepdims=True)
        min_H      = tf.math.reduce_min(energy, axis=2, keepdims=True)
        energy     = (energy - min_H) / ((max_H - min_H) + 1e-8)
        attention  = tf.nn.softmax(energy)
        out = (self.gamma
               * tf.expand_dims(
                   tf.matmul(tf.squeeze(x, axis=-1), attention, transpose_b=True),
                   axis=3)
               + x)
        return out


class MyAttention(Model):
    def __init__(self, num_ch, num_t):
        super(MyAttention, self).__init__()
        self.num_ch = num_ch
        self.num_t  = num_t
        self.CA     = Channel_attention(num_ch=num_ch)

    def get_config(self):
        config = super().get_config()
        config.update({"num_ch": self.num_ch, "num_t": self.num_t})
        return config

    def call(self, x, training=None):
        return self.CA(x)


def square(x):
    return K.square(x)

def log(x):
    return K.log(K.clip(x, min_value=1e-7, max_value=10000))


def TCN_block(input_layer, input_dimension, depth, kernel_size, filters,
              dropout, activation='relu'):
    block = Conv1D(filters, kernel_size=kernel_size, dilation_rate=1,
                   activation='linear', padding='causal',
                   kernel_initializer='he_uniform')(input_layer)
    block = BatchNormalization()(block)
    block = Activation(activation)(block)
    block = Dropout(dropout)(block)
    block = Conv1D(filters, kernel_size=kernel_size, dilation_rate=1,
                   activation='linear', padding='causal',
                   kernel_initializer='he_uniform')(block)
    block = BatchNormalization()(block)
    block = Activation(activation)(block)
    block = Dropout(dropout)(block)
    if input_dimension != filters:
        conv  = Conv1D(filters, kernel_size=1, padding='same')(input_layer)
        added = Add()([block, conv])
    else:
        added = Add()([block, input_layer])
    out = Activation(activation)(added)

    for i in range(depth - 1):
        block = Conv1D(filters, kernel_size=kernel_size, dilation_rate=2 ** (i + 1),
                       activation='linear', padding='causal',
                       kernel_initializer='he_uniform')(out)
        block = BatchNormalization()(block)
        block = Activation(activation)(block)
        block = Dropout(dropout)(block)
        block = Conv1D(filters, kernel_size=kernel_size, dilation_rate=2 ** (i + 1),
                       activation='linear', padding='causal',
                       kernel_initializer='he_uniform')(block)
        block = BatchNormalization()(block)
        block = Activation(activation)(block)
        block = Dropout(dropout)(block)
        added = Add()([block, out])
        out   = Activation(activation)(added)
    return out


def cbam_block_1d(input_feature, ratio=8, residual=False):
    cbam_feature = channel_attention_1d(input_feature, ratio)
    cbam_feature = spatial_attention_1d(cbam_feature)
    if residual:
        cbam_feature = Add()([input_feature, cbam_feature])
    return cbam_feature


def channel_attention_1d(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    reduced = max(1, channel // ratio)
    shared_layer_one = Dense(reduced, activation='relu',
                             kernel_initializer='he_normal',
                             use_bias=True, bias_initializer='zeros')
    shared_layer_two = Dense(channel, kernel_initializer='he_normal',
                             use_bias=True, bias_initializer='zeros')

    avg_pool = GlobalAveragePooling1D()(input_feature)
    avg_pool = Reshape((1, channel))(avg_pool)
    avg_pool = shared_layer_one(avg_pool)
    avg_pool = shared_layer_two(avg_pool)

    max_pool = GlobalMaxPooling1D()(input_feature)
    max_pool = Reshape((1, channel))(max_pool)
    max_pool = shared_layer_one(max_pool)
    max_pool = shared_layer_two(max_pool)

    cbam_feature = Add()([avg_pool, max_pool])
    cbam_feature = Activation('sigmoid')(cbam_feature)
    return Multiply()([input_feature, cbam_feature])


def spatial_attention_1d(input_feature):
    avg_pool     = Lambda(lambda x: K.mean(x, axis=2, keepdims=True))(input_feature)
    max_pool     = Lambda(lambda x: K.max(x, axis=2,  keepdims=True))(input_feature)
    concat       = Concatenate(axis=2)([avg_pool, max_pool])
    cbam_feature = Conv1D(filters=1, kernel_size=7, padding='same',
                          activation='sigmoid', kernel_initializer='he_normal',
                          use_bias=False)(concat)
    return Multiply()([input_feature, cbam_feature])


class GatedAttentionFusion(tf.keras.layers.Layer):
    def __init__(self, fused_dim=64):
        super().__init__()
        self.fused_dim  = fused_dim
        self.proj1      = tf.keras.layers.Dense(fused_dim)
        self.proj2      = tf.keras.layers.Dense(fused_dim)
        self.attn_dense = tf.keras.layers.Dense(fused_dim, activation='tanh')
        self.gate_score = tf.keras.layers.Dense(1)

    def get_config(self):
        config = super().get_config()
        config.update({"fused_dim": self.fused_dim})
        return config

    def call(self, x1, x2):
        p1      = self.proj1(x1)
        p2      = self.proj2(x2)
        stacked = tf.stack([p1, p2], axis=2)           # (B, T, 2, D)
        attn_h  = self.attn_dense(stacked)
        attn_s  = self.gate_score(attn_h)
        weights = tf.nn.softmax(attn_s, axis=2)
        fused   = tf.reduce_sum(weights * stacked, axis=2)
        return fused


# =============================================================
# Cell - Build the Model
# =============================================================
temporal_dimension = X_train_new.shape[1]   # 500
num_channels       = X_train_new.shape[2]   # 3
num_classes        = 2                      # Left hand / Right hand

input_main  = Input(shape=(temporal_dimension, num_channels), name="input_main")
input_theta = Input(shape=(temporal_dimension, num_channels), name="input_theta")
input_alpha = Input(shape=(temporal_dimension, num_channels), name="input_alpha")
input_beta  = Input(shape=(temporal_dimension, num_channels), name="input_beta")

# --- Main Branch ---
x_main = Reshape((num_channels, temporal_dimension, 1))(input_main)
x_main = MyAttention(num_ch=num_channels, num_t=temporal_dimension)(x_main)
x      = Reshape((temporal_dimension, num_channels))(x_main)

x1     = TCN_block(input_layer=input_main, input_dimension=num_channels, depth=2,
                   kernel_size=4, filters=16, dropout=0.3, activation='elu')

x2     = tf.keras.layers.Concatenate()([x, x1])
x3     = cbam_block_1d(x2)

# --- Theta Branch ---
x_theta = Reshape((num_channels, temporal_dimension, 1))(input_theta)
x_theta = MyAttention(num_ch=num_channels, num_t=temporal_dimension)(x_theta)
xt      = Reshape((temporal_dimension, num_channels))(x_theta)

# --- Alpha Branch ---
x_alpha = Reshape((num_channels, temporal_dimension, 1))(input_alpha)
x_alpha = MyAttention(num_ch=num_channels, num_t=temporal_dimension)(x_alpha)
xa      = Reshape((temporal_dimension, num_channels))(x_alpha)

# --- Beta Branch ---
x_beta = Reshape((num_channels, temporal_dimension, 1))(input_beta)
x_beta = MyAttention(num_ch=num_channels, num_t=temporal_dimension)(x_beta)
xb     = Reshape((temporal_dimension, num_channels))(x_beta)

# --- Sub-band Fusion ---
x2_sub = tf.keras.layers.Concatenate()([xa, xt, xb])   # → (N, 500, 9)
x31    = cbam_block_1d(x2_sub)

# --- Gated Attention Fusion (main + sub-band) ---
fused_dim = int(x3.shape[-1]) + int(x31.shape[-1])
fused     = GatedAttentionFusion(fused_dim=fused_dim)(x3, x31)

# --- Output Block ---
out = Conv1D(32, 5, activation='relu', padding='same')(fused)
out = MaxPooling1D(2)(out)
out = BatchNormalization()(out)
out = Dropout(0.4)(out)

out = Conv1D(64, 3, activation='relu', padding='same')(out)
out = BatchNormalization()(out)
out = GlobalAveragePooling1D()(out)
out = Dropout(0.3)(out)

out = Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.002))(out)
out = Dense(units=num_classes)(out)
out = Activation('softmax')(out)

model = Model(
    inputs=[input_main, input_theta, input_alpha, input_beta],
    outputs=out
)
model.build(((None, temporal_dimension, num_channels)))
model.summary()


# =============================================================
# Cell - Compile & Train
# =============================================================
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=1e-4),
    metrics=['accuracy',
             tf.keras.metrics.Precision(),
             tf.keras.metrics.Recall()]
)


class ThresholdStopping(Callback):
    def __init__(self, monitor='val_loss', threshold=0.025):
        super().__init__()
        self.monitor   = monitor
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is not None and current <= self.threshold:
            print(f"\nEpoch {epoch+1}: {self.monitor}={current:.4f} ≤ {self.threshold}, stopping.")
            self.model.stop_training = True


results_dir = "STSPecResult_2b_S1"
os.makedirs(results_dir, exist_ok=True)

checkpoint = [
    ThresholdStopping(monitor='val_loss', threshold=0.025),
    ModelCheckpoint(
        filepath=os.path.join(results_dir, 'STSPec_best_model_2b_S1.h5'),
        monitor='val_loss',
        save_best_only=True,
        mode='min'
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=20, min_lr=1e-5)
]

print("\nStarting training …")
start_time = time.time()
history = model.fit(
    [X_train_new, X_train_theta, X_train_alpha, X_train_beta],
    y_train_new_oh,
    epochs=1200,
    batch_size=64,
    validation_data=(
        [X_val_new, X_val_theta, X_val_alpha, X_val_beta],
        y_val_new_oh
    ),
    callbacks=checkpoint,
    shuffle=True,
    verbose=2
)
total_time = time.time() - start_time
print(f"\nTraining completed in {int(total_time // 60)}m {total_time % 60:.1f}s")


# =============================================================
# Cell - Evaluation
# =============================================================
def evaluate_split(X, y_true, split_name="Test"):
    y_prob = model.predict(X)
    y_pred = np.argmax(y_prob, axis=1)
    acc    = accuracy_score(y_true, y_pred)
    prec   = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec    = recall_score(y_true, y_pred,    average='weighted', zero_division=0)
    f1     = f1_score(y_true, y_pred,        average='weighted', zero_division=0)
    kappa  = cohen_kappa_score(y_true, y_pred)
    print(f"\n*** {split_name} Metrics ***")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"Kappa    : {kappa:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4))
    return confusion_matrix(y_true, y_pred)


cm_train = evaluate_split(
    [X_train_new, X_train_theta, X_train_alpha, X_train_beta], y_train_new, "Training")
cm_val   = evaluate_split(
    [X_val_new,   X_val_theta,   X_val_alpha,   X_val_beta],   y_val_new,   "Validation")
cm_test  = evaluate_split(
    [X_test_new,  X_test_theta,  X_test_alpha,  X_test_beta],  y_test_new,  "Test")


# =============================================================
# Cell - Confusion Matrices
# =============================================================
classes_labels = ['Left hand', 'Right hand']


def plot_confusion_matrix_percent(cm, class_labels, title, save_path):
    cm       = cm.astype('float')
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_pct   = (cm / row_sums) * 100
    labels   = np.array([[f"{v:.1f}%" for v in row] for row in cm_pct])

    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm_pct, interpolation='nearest', cmap=plt.cm.Greens)
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(len(class_labels)),
        yticks=np.arange(len(class_labels)),
        xticklabels=class_labels,
        yticklabels=class_labels,
        xlabel='Predicted label',
        ylabel='True label'
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    thresh = cm_pct.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, labels[i, j], ha="center", va="center",
                    color="white" if cm_pct[i, j] > thresh else "black")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(save_path)
    plt.close(fig)


for cm, name in zip([cm_train, cm_val, cm_test], ["Train", "Val", "Test"]):
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes_labels)
    fig, ax = plt.subplots(figsize=(4, 4))
    disp.plot(cmap=plt.cm.Blues, ax=ax)
    fig.tight_layout()
    fig.savefig(os.path.join(results_dir, f"{name}_confusion_matrix_2b_S1.png"))
    plt.close(fig)

for cm, name in zip([cm_train, cm_val, cm_test], ["Train", "Val", "Test"]):
    plot_confusion_matrix_percent(
        cm, classes_labels,
        f"{name} 2b S1 Confusion Matrix (Percentages)",
        os.path.join(results_dir, f"{name}_confusion_matrix_percent_2b_S1.png")
    )


# =============================================================
# Cell - Learning Curves
# =============================================================
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

axs[0].plot(history.history['accuracy'])
axs[0].plot(history.history['val_accuracy'])
axs[0].set_title('Accuracy')
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Accuracy')
axs[0].legend(['Train', 'Val'], loc='upper left')

axs[1].plot(history.history['loss'])
axs[1].plot(history.history['val_loss'])
axs[1].set_title('Loss')
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Loss')
axs[1].legend(['Train', 'Val'], loc='upper left')

fig.tight_layout()
fig.savefig(os.path.join(results_dir, "learning_curves_STSPec_2b_S1.png"))
plt.show()


# =============================================================
# Cell - t-SNE Visualisation
# =============================================================
def plot_and_save_tsne(features_input, labels, split_name, save_path, class_labels):
    preds  = model.predict(features_input)
    X_flat = preds.reshape(preds.shape[0], -1)
    tsne   = TSNE(n_components=2, random_state=42)
    emb    = tsne.fit_transform(X_flat)
    uniq   = np.unique(labels)
    colors = plt.cm.tab10(np.linspace(0, 1, len(uniq)))

    fig, ax = plt.subplots(figsize=(8, 5))
    for i, cls in enumerate(uniq):
        idx = labels == cls
        ax.scatter(emb[idx, 0], emb[idx, 1],
                   color=colors[i], label=class_labels[int(cls)], s=40)
    ax.set_title(f"t-SNE — {split_name}")
    ax.set_xlabel('t-SNE Feature 1')
    ax.set_ylabel('t-SNE Feature 2')
    ax.legend(title='Classes', loc='upper left')
    fig.tight_layout()
    fig.savefig(save_path)
    plt.show()
    return emb


plot_and_save_tsne(
    [X_train_new, X_train_theta, X_train_alpha, X_train_beta],
    y_train_new, "Train",
    os.path.join(results_dir, "Train_STSPecTSNE_2b_S1.png"),
    classes_labels
)
plot_and_save_tsne(
    [X_val_new, X_val_theta, X_val_alpha, X_val_beta],
    y_val_new, "Validation",
    os.path.join(results_dir, "Val_STSPecTSNE_2b_S1.png"),
    classes_labels
)
plot_and_save_tsne(
    [X_test_new, X_test_theta, X_test_alpha, X_test_beta],
    y_test_new, "Test",
    os.path.join(results_dir, "Test_STSPecTSNE_2b_S1.png"),
    classes_labels
)


# =============================================================
# Cell - Save training history
# =============================================================
np.savez(
    os.path.join(results_dir, "training_history_2b_S1.npz"),
    **history.history
)
print("\nAll results saved to", results_dir)